# Setup

# This file requires using the environment created from environment-gtfparse.yml

In [1]:
# Imports

import polars as pl
import duckdb
from pathlib import Path
import math
import os
import numpy as np
from tqdm.notebook import tqdm 

For this analysis you will need to obtain the COSMIC genome screening files as well as the COSMIC mutation classification file.
This analysis was based on the v99 release of COSMIC database
You can find the files and the license information at
https://cancer.sanger.ac.uk/cosmic/download/cosmic

In [ ]:
# Paths
project_dir = Path("/data/teamgdansk/mwaleron/carmen-analysis")

data_dir = project_dir.joinpath("data")
temp_dir = project_dir.joinpath("temp")

db_main_file = data_dir.joinpath("carmen-main.parquet")

data_subs_dir = data_dir.joinpath("subsidiary-files")
data_pub_dir = data_dir.joinpath("to-be-published")

braun_dataset_mut = data_subs_dir.joinpath("Braun_mutations_hg38_with_epitope_contigs.tsv")


contig_scaffold_list_file = data_pub_dir.joinpath("scaff_all_expanded.tsv")
contig_peptide_list_file = data_pub_dir.joinpath("contig_unique_peptide_list.tsv")
contig_hotspot_score_file = data_subs_dir.joinpath("sorted_normalized_contig_score.parquet")

main_samples_file = data_dir.joinpath("main-output-table-1.tsv")
main_samples_file_2 = data_dir.joinpath("main-output-table-2.tsv")
main_samples_file_3 = data_subs_dir.joinpath("emilia_umap_with_ids.parquet")

scaff_10_pep_hla_file = temp_dir.joinpath("Braun_hg38_pep_hla_scaff_10gw.parquet")
scaff_20_pep_hla_file = temp_dir.joinpath("Braun_hg38_pep_hla_scaff_20gw.parquet")
scaff_10_prom_file = temp_dir.joinpath("promiscuity_scaffolds_gw_of_10.csv")
scaff_20_prom_file = temp_dir.joinpath("promiscuity_scaffolds_gw_of_20.csv")

In [ ]:
contig_scaffold_list = pl.read_csv(
    contig_scaffold_list_file,
    separator='\t')
contig_peptide_list = pl.read_csv(
    contig_peptide_list_file, 
    separator='\t')

In [3]:
pep_main=pl.read_parquet(db_main_file)

In [ ]:
patient_mutations_2_hg38loo = pl.read_csv(braun_dataset_mut, separator="\t", ignore_errors=True)
patient_mutations_2_hg38loo_e = patient_mutations_2_hg38loo.with_row_index("mutid")
patient_mutations_2_hg38loo_e.write_parquet("braun_mutations_with_idx.parquet")

In [ ]:
pogo_peptides = pl.read_parquet("/data/teamgdansk/mwaleron/carmen-analysis/data/carmen-mapped-protein-annotations-pogo.parquet")

In [ ]:
peptides_in_mutations2 = duckdb.sql('''
                              select *
                              from pogo_peptides p, patient_mutations_2_hg38loo_e m
                              where (p.Chromosome = m.Chromosome)
                                 and(
                                    (m.Start_position between p.Gene_start and p.Gene_end)
                                 or (m.End_position between p.Gene_start and p.Gene_end)
                                 )                                
                                 ''')

In [ ]:
patient_peptides_in_mutations_2 = peptides_in_mutations2.pl()
patient_peptides_in_mutations_2.write_parquet("patient_peptides_in_mutations_2_before_grouping.parquet")

In [ ]:
pep_haplotype = pep_main.group_by("Peptide").agg(pl.col("Haplotype").str.split(",").explode().unique())

In [ ]:
narrow_peptides_haplotype = patient_peptides_in_mutations_2\
    .join(pep_haplotype, on="Peptide")\
    .group_by("mutid")\
    .agg(pl.col("Haplotype").explode().unique().alias("Haplotype_narrow"))\
    .with_columns(Unique_haplotypes_narrow=pl.col("Haplotype_narrow").list.len())
narrow_peptides_mut = patient_peptides_in_mutations_2\
    .group_by("mutid")\
    .agg(pl.col("Peptide").unique(), pl.col("Gene_start").min(), pl.col("Gene_end").max(), pl.col("Chromosome").last())\
    .with_columns(unique_peptides = pl.col("Peptide").list.len(), left_broad = pl.col("Gene_start")-15, right_broad = pl.col("Gene_end") + 15)

In [ ]:
narrow_peptides_mut.join(narrow_peptides_haplotype, on="mutid").rename(
    {"Peptide":"Peptide_list_narrow",
     "Gene_start":"Gene_start_narrow",
     "Gene_end":"Gene_end_narrow",
     "unique_peptides":"Unique_peptides_narrow",
     "left_broad":"Gene_start_broad",
     "right_broad":"Gene_end_broad"}
).write_parquet("narrow_mutation_wo_prom.parquet")

In [ ]:
broad_peptides_mut = duckdb.sql('''
                              select *
                              from pogo_peptides p, narrow_peptides_mut m
                              where (p.Chromosome = m.Chromosome)
                                 and(
                                    (p.Gene_start between m.left_broad and m.right_broad)
                                 or (p.Gene_end between m.left_broad and m.right_broad)
                                 )                                
                                 ''').pl()

In [ ]:
broad_peptides_haplotype = broad_peptides_mut\
    .join(pep_haplotype, on="Peptide")\
    .group_by("mutid")\
    .agg(pl.col("Haplotype").explode().unique().alias("Haplotype_broad"))\
    .with_columns(Unique_haplotypes_broad=pl.col("Haplotype_broad").list.len())

In [ ]:
broad_peptides_just_uniq = broad_peptides_mut\
    .group_by("mutid")\
    .agg(pl.col("Peptide").unique().alias("Peptide_broad"))\
    .with_columns(broad_unique_peptides = pl.col("Peptide_broad").list.len())

In [ ]:
narrow_peptides_mut.join(narrow_peptides_haplotype, on="mutid").rename(
    {"Peptide":"Peptide_list_narrow",
     "Gene_start":"Gene_start_narrow",
     "Gene_end":"Gene_end_narrow",
     "unique_peptides":"Unique_peptides_narrow",
     "left_broad":"Gene_start_broad",
     "right_broad":"Gene_end_broad"}
).join(broad_peptides_just_uniq.join(broad_peptides_haplotype, on="mutid").rename(
    {"broad_unique_peptides" : "Unique_peptides_broad"}
), on="mutid").write_parquet("narrow_and_broad_braun_mutation_wo_prom.parquet")

In [ ]:
peptides_of_10gw_scaffolds = \
    contig_scaffold_list\
        .join(
            contig_peptide_list, 
            left_on="Contigs", 
            right_on="contig")\
        .filter(pl.col("gap_width")==10)\
        .group_by("Id").agg(pl.col("pep_list").str.concat(",").str.split(","))

In [20]:
peptides_of_10gw_scaffolds\
    .explode("pep_list")\
    .join(
        pep_main.select("Peptide", "Haplotype"), 
        left_on="pep_list", 
        right_on="Peptide")\
    .with_columns(pl.col("Haplotype").str.split(","))\
    .explode("Haplotype")\
    .unique()\
    .group_by("Id")\
        .agg(pl.col("Haplotype"), pl.col("pep_list"))\
    .with_columns(
        unique_peptides=pl.col("pep_list").list.len(),
        n_unique_HLA=pl.col("Haplotype").list.len()
    ).write_parquet("Braun_hg38_pep_hla_scaff_10gw.parquet")

In [ ]:
peptides_of_20gw_scaffolds = \
    contig_scaffold_list\
        .join(
            contig_peptide_list, 
            left_on="Contigs", 
            right_on="contig")\
        .filter(pl.col("gap_width")==20)\
        .group_by("Id").agg(pl.col("pep_list").str.concat(",").str.split(","))

In [22]:
peptides_of_20gw_scaffolds\
    .explode("pep_list")\
    .join(
        pep_main.select("Peptide", "Haplotype"), 
        left_on="pep_list", 
        right_on="Peptide")\
    .with_columns(pl.col("Haplotype").str.split(","))\
    .explode("Haplotype")\
    .unique()\
    .group_by("Id")\
        .agg(pl.col("Haplotype"), pl.col("pep_list"))\
    .with_columns(
        unique_peptides=pl.col("pep_list").list.len(),
        n_unique_HLA=pl.col("Haplotype").list.len()
    ).write_parquet("Braun_hg38_pep_hla_scaff_20gw.parquet")

# Jump here

In [5]:
braun_full = pl.read_parquet(data_subs_dir.joinpath("braun_mutations_with_idx.parquet"))

In [7]:
braun_to_score = pl.read_parquet(data_subs_dir.joinpath("narrow_and_broad_braun_mutation_wo_prom.parquet"))

In [10]:
emilia_umap_with_ids = pl.read_parquet(main_samples_file_3)
xallcen = (emilia_umap_with_ids["x"].min() + emilia_umap_with_ids["x"].max() )/2
yallcen = (emilia_umap_with_ids["y"].min() + emilia_umap_with_ids["y"].max() )/2

In [11]:
def promiscuity_peptide_list(input):
    scorebase = emilia_umap_with_ids\
        .with_columns(
        scafres = pl.col("Peptides").list.set_intersection(input
        ))\
        .with_columns(alen = pl.col("scafres").list.len()).sort("alen")
    xy = scorebase.filter(pl.col("alen")>0).select("x","y")
    count = len(xy)
    if count == 0 :
        return 0
    xcen = xy["x"].sum()/count
    ycen = xy["y"].sum()/count
    lth = np.sqrt( np.power(xallcen-xcen,2) + np.power(yallcen-ycen,2))
    popcov_but_sqrt = count / math.sqrt(1+lth)
    return popcov_but_sqrt

The next parquet is produced by calcualte_immune_scorer_3.py (just run it once)

In [ ]:
braun_prom_overlap = pl.read_parquet("braun_prom_just_overlap.parquet")

In [10]:
braun_prom_overlap.mean()

mutid,Peptide_list_narrow,Gene_start_narrow,Gene_end_narrow,Chromosome,Unique_peptides_narrow,Gene_start_broad,Gene_end_broad,Haplotype_narrow,Unique_haplotypes_narrow,Peptide_broad,Unique_peptides_broad,Haplotype_broad,Unique_haplotypes_broad,Promiscuity_narrow,Promiscuity_broad
f64,list[str],f64,f64,str,f64,f64,f64,list[str],f64,list[str],f64,list[str],f64,f64,f64
31189.14571,null,7.1988e7,7.1993e7,null,2.032894,7.1988e7,7.1993e7,null,20.174263,null,6.549462,null,40.082525,4.692038,17.72858


In [12]:
braun_full.join(braun_prom_overlap, on="mutid", how="left").with_columns(pl.col("Peptide_list_narrow").list.join(","), pl.col("Haplotype_narrow").list.join(","), pl.col("Peptide_broad").list.join(","), pl.col("Haplotype_broad").list.join(",")).write_csv("braun_mutations_alternative_scoring_narrow_broad.tsv", separator="\t")

/tmp/ipykernel_3688619/4168136393.py:1: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  braun_full.join(braun_prom_overlap, on="mutid", how="left").with_columns(pl.col("Peptide_list_narrow").list.join(","), pl.col("Haplotype_narrow").list.join(","), pl.col("Peptide_broad").list.join(","), pl.col("Haplotype_broad").list.join(",")).write_csv("braun_mutations_alternative_scoring_narrow_broad.tsv", separator="\t")


This is also generated by calculate immune scorer 3

In [2]:
braun_prom_all = pl.read_parquet("braun_prom_all.parquet")

In [3]:
braun_prom_all

mutid,Peptide_list_narrow,Gene_start_narrow,Gene_end_narrow,Chromosome,Unique_peptides_narrow,Gene_start_broad,Gene_end_broad,Haplotype_narrow,Unique_haplotypes_narrow,Peptide_broad,Unique_peptides_broad,Haplotype_broad,Unique_haplotypes_broad,Promiscuity_narrow,Promiscuity_broad,Sample_ID,Chromosome_right,Start_position,End_position,Variant_Classification,Variant_Type,cDNA_Change,Codon_Change,Protein_Change,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,Tumor_ref_count,Tumor_alt_count,gene_name,SUBJID,contig
u32,list[str],i64,i64,str,u32,i64,i64,list[str],u32,list[str],u32,list[str],u32,f64,f64,str,str,i64,i64,str,str,str,str,str,str,str,str,i64,i64,str,str,str
0,"[""KEKEDVPVEM""]",52579172,52586427,"""3""",1,52579157,52586442,"[""A*02:17"", ""B*41:01"", … ""C*17:01""]",6,"[""KEKEDVPVEM"", ""SRAEDNFNL""]",2,"[""B*08:01"", ""B*40:02"", … ""C*17:01""]",17,1.077932,2.129851,"""RCC_15-Tumor-SM-A46EE""","""3""",52579179,52579179,"""Silent""","""SNP""","""c.3408G>T""","""c.(3406-3408)gtG>gtT""","""p.V1136V""","""C""","""C""","""A""",44,53,"""PBRM1""","""RCC_15""","""C48753"""
1,"[""RRLDIVRSLYE"", ""RRLDIVRSLY"", … ""RRLDIVRSL""]",10149842,10149881,"""3""",5,10149827,10149896,"[""C*02:02:02"", ""A*01"", … ""C*14:02""]",86,"[""RRLDIVRSL"", ""NYRRLDIVR"", … ""VRSLVKPENY""]",8,"[""C*12:03"", ""B*27:13"", … ""A*01:01""]",86,50.871353,50.967784,"""EA698866""","""3""",10149856,10149856,"""Missense_Mutation""","""SNP""","""c.533T>A""","""c.(532-534)cTg>cAg""","""p.L178Q""","""T""","""T""","""A""",92,43,"""VHL""","""RCC25-732""","""C43929"""
2,"[""SREPSQVIF""]",10142048,10142075,"""3""",1,10142033,10142090,"[""C*05:01"", ""B*44:02"", … ""A*01:01""]",16,"[""AGRPRPVL"", ""SREPSQVIF"", … ""SPRVVLPVWL""]",7,"[""B*44:02"", ""B*55"", … ""B*40:01""]",63,1.674095,27.258109,"""EA698881""","""3""",10142055,10142055,"""Nonsense_Mutation""","""SNP""","""c.208G>T""","""c.(208-210)Gag>Tag""","""p.E70*""","""G""","""G""","""T""",155,101,"""VHL""","""RCC25-535""","""C43924"""
6,"[""QVLEGHVLSEA""]",38074866,38074899,"""2""",1,38074851,38074914,"[""C*07:02"", ""A*02:01"", … ""B*44:02""]",5,"[""SEARELVALLV"", ""SEARELVALL"", … ""QPRSRQVL""]",8,"[""B*44:02"", ""C*07:01"", … ""A*01:01:01""]",53,0.0,9.140501,"""EA700366""","""2""",38074887,38074887,"""Missense_Mutation""","""SNP""","""c.502G>T""","""c.(502-504)Ggc>Tgc""","""p.G168C""","""C""","""C""","""A""",199,14,"""CYP1B1""","""RCC25-345""","""C28244"""
7,"[""LAAAVHKEM""]",53201600,53201627,"""X""",1,53201585,53201642,"[""C*16:01"", ""B*40:01"", … ""C*03:04""]",4,"[""LAAAVHKEM"", ""MVQEERRLR""]",2,"[""A*32:01"", ""C*03:04"", … ""C*08:02""]",10,1.634825,3.376688,"""EA700362""","""X""",53201604,53201606,"""In_Frame_Del""","""DEL""","""c.2005_2007delGAG""","""c.(2005-2007)gagdel""","""p.E669del""","""CTC""","""CTC""","""-""",38,31,"""KDM5C""","""RCC25-486""","""C242184"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
62640,"[""VSRIKENGAAAL"", ""SRIKENGAAAL"", ""RIKENGAAA""]",70388505,70388541,"""14""",3,70388490,70388556,"[null, ""A*02:06"", … ""A*24:02""]",28,"[""RIKENGAAA"", ""QEGDKILSV"", … ""RLQEGDKILSV""]",7,"[""A*01:01"", ""B*18:01"", … ""B*51:01""]",72,1.518688,19.179558,"""EA699911""","""14""",70388526,70388526,"""Missense_Mutation""","""SNP""","""c.145G>A""","""c.(145-147)Gaa>Aaa""","""p.E49K""","""C""","""C""","""T""",130,58,"""SYNJ2BP""","""RCC25-116""","""C167583"""
62645,"[""ILPKKSWHV""]",54159662,54159689,"""19""",1,54159647,54159704,[null],1,"[""ILPKKSWHV""]",1,[null],1,0.447944,0.447944,"""EA700076""","""19""",54159680,54159680,"""IGR""","""SNP""",null,null,null,"""T""","""T""","""C""",452,18,"""TMC4""","""RCC25-931""","""C223532"""
62656,"[""VATGVISTL"", ""VLQVATGVIST""]",66089656,66089692,"""7""",2,66089641,66089707,"[""C*14:02"", ""C*02:02"", … ""A*29:02""]",34,"[""VLQVATGVIST"", ""DTMSAVLQV"", ""VATGVISTL""]",3,"[""B*15:11"", ""A*68:01"", … ""C*05:01""]",37,3.299354,3.299354,"""EA700541""","""7""",66089677,66089677,"""Missense_Mutation""","""SNP""","""c.349C>T""","""

# scaffold scoring

These parquet files are generated in step 11 by calculate_immune_scorer.py and calculate_immune_scorer_2.py respetively.

In [22]:
scaffold10_scores = pl.scan_parquet("/data/teamgdansk/mwaleron/carmen-analysis/temp/parquetspam/rescored_scaf10_slice_*.parquet")

In [ ]:
scaffold10_scores.sort("Id").drop("pep_list").sink_csv(scaff_10_prom_file)

In [7]:
scaffold20_scores = pl.scan_parquet("/data/teamgdansk/mwaleron/carmen-analysis/temp/parquetspam/rescored_scaf20_slice_*.parquet")

In [ ]:
scaffold20_scores.sort("Id").drop("pep_list").sink_csv(scaff_20_prom_file)

In [13]:
braunmut = pl.read_csv(braun_dataset_mut, separator="\t", ignore_errors=True)

In [8]:
braunmut = pl.read_parquet(data_subs_dir.joinpath("braun_mutations_with_idx.parquet"))

In [20]:
braunmut

Sample_ID,Chromosome,Start_position,End_position,Variant_Classification,Variant_Type,cDNA_Change,Codon_Change,Protein_Change,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,Tumor_ref_count,Tumor_alt_count,gene_name,SUBJID,contig
str,str,i64,i64,str,str,str,str,str,str,str,str,i64,i64,str,str,str
"""RCC_15-Tumor-SM-A46EE""","""3""",52579179,52579179,"""Silent""","""SNP""","""c.3408G>T""","""c.(3406-3408)gtG>gtT""","""p.V1136V""","""C""","""C""","""A""",44,53,"""PBRM1""","""RCC_15""","""C48753"""
"""EA698866""","""3""",10149856,10149856,"""Missense_Mutation""","""SNP""","""c.533T>A""","""c.(532-534)cTg>cAg""","""p.L178Q""","""T""","""T""","""A""",92,43,"""VHL""","""RCC25-732""","""C43929"""
"""EA698881""","""3""",10142055,10142055,"""Nonsense_Mutation""","""SNP""","""c.208G>T""","""c.(208-210)Gag>Tag""","""p.E70*""","""G""","""G""","""T""",155,101,"""VHL""","""RCC25-535""","""C43924"""
"""EA700340""","""3""",10149807,10149807,"""Missense_Mutation""","""SNP""","""c.484T>C""","""c.(484-486)Tgc>Cgc""","""p.C162R""","""T""","""T""","""C""",23,7,"""VHL""","""RCC25-181""",null
"""RCC_52-Tumor-SM-A46EU""","""2""",178632229,178632229,"""Missense_Mutation""","""SNP""","""c.38742C>G""","""c.(38740-38742)ttC>ttG""","""p.F12914L""","""G""","""G""","""C""",62,23,"""TTN""","""RCC_52""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""EA700525""","""6""",155314319,155314319,"""Missense_Mutation""","""SNP""","""c.110T>C""","""c.(109-111)tTc>tCc""","""p.F37S""","""A""","""A""","""G""",90,40,"""TFB1M""","""RCC25-915""",null
"""EA700647""","""16""",696895,696899,"""Frame_Shift_Del""","""DEL""","""c.507_511delGGGCT""","""c.(505-513)gagggcttcfs""","""p.EGF169fs""","""AGCCC""","""AGCCC""","""-""",547,41,"""FBXL16""","""RCC25-671""",null
"""EA700541""","""7""",66089677,66089677,"""Missense_Mutation""","""SNP""","""c.349C>T""","""c.(349-351)Cgt>Tgt""","""p.R117C""","""C""","""C""","""T""",107,4,"""AC068533.7""","""RCC25-284""","""C95434"""


In [9]:
braunmut_scaffold10 = duckdb.sql('''
                                 select * 
                                 from braunmut b, scaff10 c
                                 where c.Chromosome = b.Chromosome
                                 and(
                                     (b.Start_position between c.Gene_start and c.Gene_end)
                                   or(b.End_position between c.Gene_start and c.Gene_end)
                                 )
                                 ''').pl()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
braunmut_scaffold10.write_csv("Braun_hg38_mutations_in_scaffolds_10gw.csv")

In [32]:
braunmut_scaffold10.select(pl.len())

len
u32
20638


In [31]:
braunmut_scaffold10.group_by(["Sample_ID", "Chromosome", "Start_position", "Id"]).agg(pl.all())

Sample_ID,Chromosome,Start_position,Id,End_position,Variant_Classification,Variant_Type,cDNA_Change,Codon_Change,Protein_Change,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,Tumor_ref_count,Tumor_alt_count,gene_name,SUBJID,contig,Chromosome_1,Gene_start,Gene_end,Multi_contig,gap_width
str,str,i64,str,list[i64],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[i64],list[i64],list[str],list[str],list[str],list[str],list[i64],list[i64],list[str],list[i64]
"""EA700252""","""1""",218405195,"""S_10_18673""",[218405195],"[""Missense_Mutation""]","[""SNP""]","[""c.373C>T""]","[""c.(373-375)Ccc>Tcc""]","[""p.P125S""]","[""C""]","[""C""]","[""T""]",[74],[4],"[""TGFB2""]","[""RCC25-753""]","[""C22430""]","[""1""]",[218363411],[218405218],"[""N""]",[10]
"""EA699820""","""3""",49373315,"""S_10_39520""",[49373315],"[""Intron""]","[""SNP""]",[null],[null],[null],"[""A""]","[""A""]","[""G""]",[46],[4],"[""RHOA""]","[""RCC25-532""]","[""C47646""]","[""3""]",[49360323],[49375520],"[""N""]",[10]
"""EA699642""","""14""",63283169,"""S_10_138977""",[63283169],"[""Nonsense_Mutation""]","[""SNP""]","[""c.451A>T""]","[""c.(451-453)Aaa>Taa""]","[""p.K151*""]","[""A""]","[""A""]","[""T""]",[20],[9],"[""RHOJ""]","[""RCC25-728""]","[""C166753""]","[""14""]",[63283163],[63283189],"[""N""]",[10]
"""EA698479""","""2""",159186856,"""S_10_29946""",[159186878],"[""Intron""]","[""DEL""]",[null],[null],[null],"[""AGATCTGTCTCTGATTGTTTCCA""]","[""AGATCTGTCTCTGATTGTTTCCA""]","[""-""]",[130],[4],"[""TANC1""]","[""RCC25-883""]","[""C36217""]","[""2""]",[159185822],[159186952],"[""Y""]",[10]
"""EA698908""","""11""",61908090,"""S_10_114118""",[61908090],"[""Silent""]","[""SNP""]","[""c.228C>T""]","[""c.(226-228)tcC>tcT""]","[""p.S76S""]","[""G""]","[""G""]","[""A""]",[288],[6],"[""RAB3IL1""]","[""RCC25-418""]","[""C136868""]","[""11""]",[61907604],[61908110],"[""Y""]",[10]
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""EA698359""","""12""",14542121,"""S_10_122458""",[14542122],"[""Intron""]","[""INS""]",[null],[null],[null],"[""-""]","[""-""]","[""C""]",[19],[4],"[""PLBD1""]","[""RCC25-667""]","[""C146859""]","[""12""]",[14540881],[14542218],"[""N""]",[10]
"""EA699724""","""15""",78490646,"""S_10_148330""",[78490651],"[""In_Frame_Del""]","[""DEL""]","[""c.2209_2214delATTGAA""]","[""c.(2209-2214)attgaadel""]","[""p.IE737del""]","[""ATTGAA""]","[""ATTGAA""]","[""-""]",[31],[3],"[""IREB2""]","[""RCC25-103""]","[""C178027""]","[""15""]",[78490625],[78490672],"[""N""]",[10]
"""EA699911""","""17""",7823961,"""S_10_160677""",[7823961],"[""Silent""]","[""SNP""]","[""c.11457G>A""]","[""c.(11455-11457)ccG>ccA""]","[""p.P3819P""]","[""G""]","[""G""]","[""A""]",[137],[50],"[""DNAH2""]","[""RCC25-116""]","[""C192987""]","[""17""]",[7823950],[7823976],"[""N""]",[10]


In [23]:
braunmut_scaffold10 = pl.read_csv("Braun_hg38_mutations_in_scaffolds_10gw.csv")
braunmut_scaffold20 = pl.read_csv("Braun_hg38_mutations_in_scaffolds_20gw.csv")

In [14]:
braunpephla_scaffold10 = pl.read_parquet("Braun_hg38_pep_hla_scaff_10gw.parquet")
braunpephla_scaffold20 = pl.read_parquet("Braun_hg38_pep_hla_scaff_20gw.parquet")

In [ ]:
scaf10prom = pl.read_csv(scaff_10_prom_file)
scaf20prom = pl.read_csv("promiscuity_scaffolds_gw_of_20.csv")

In [27]:
scaf10prom

Id,popcov_but_sqrt,popcov_but_sqrt2,popcov_but_sqrt3,popcov_but_sqrt4
str,f64,f64,f64,f64
"""S_10_1""",0.510925,0.0,0.0,0.0
"""S_10_10""",0.0,0.0,0.0,0.0
"""S_10_100""",0.0,0.0,0.0,0.0
"""S_10_1000""",0.376947,0.0,0.0,0.0
"""S_10_10000""",2.281,0.0,0.0,0.0
…,…,…,…,…
"""S_10_99995""",12.144016,0.0,0.0,0.0
"""S_10_99996""",12.836341,0.0,0.0,0.0
"""S_10_99997""",1.439115,0.0,0.0,0.0


In [35]:
braunmut_scaffold20

mutid,Sample_ID,Chromosome,Start_position,End_position,Variant_Classification,Variant_Type,cDNA_Change,Codon_Change,Protein_Change,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,Tumor_ref_count,Tumor_alt_count,gene_name,SUBJID,contig,Id,Chromosome_1,Gene_start,Gene_end,Multi_contig,gap_width
u32,str,str,i64,i64,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64,str,i64
62228,"""EA700350""","""4""",142314840,142314840,"""Intron""","""DEL""",null,null,null,"""T""","""T""","""-""",89,22,"""INPP4B""","""RCC25-87""","""C65397""","""S_20_47175""","""4""",142314742,142402946,"""N""",20
62506,"""EA698327""","""19""",57844252,57844252,"""IGR""","""SNP""",null,null,null,"""A""","""A""","""G""",46,7,"""ZNF587B""","""RCC25-96""","""C224821""","""S_20_162177""","""19""",57819925,57856142,"""N""",20
62213,"""RP-1458_RCCBMS-00153-T_v4_Exom…","""19""",17191105,17191105,"""Silent""","""SNP""","""c.2697C>T""","""c.(2695-2697)acC>acT""","""p.T899T""","""C""","""C""","""T""",89,5,"""MYO9B""","""RCC10-153""","""C214812""","""S_20_154810""","""19""",17191103,17192913,"""Y""",20
60018,"""EA699646""","""18""",12971255,12971255,"""Intron""","""SNP""",null,null,null,"""A""","""A""","""G""",85,25,"""SEH1L""","""RCC25-582""","""C205907""","""S_20_148369""","""18""",12971211,12978776,"""N""",20
61726,"""EA699807""","""3""",10142115,10142115,"""Frame_Shift_Del""","""DEL""","""c.268delA""","""c.(268-270)aacfs""","""p.N90fs""","""A""","""A""","""-""",163,48,"""VHL""","""RCC25-260""","""C43925""","""S_20_31573""","""3""",10142010,10142120,"""Y""",20
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2889,"""EA699824""","""1""",11124535,11124535,"""Missense_Mutation""","""SNP""","""c.6625C>G""","""c.(6625-6627)Ctg>Gtg""","""p.L2209V""","""G""","""G""","""C""",68,16,"""MTOR""","""RCC25-379""","""C1647""","""S_20_1200""","""1""",11122114,11126721,"""Y""",20
2693,"""EA700001""","""1""",246903941,246903941,"""Intron""","""SNP""",null,null,null,"""C""","""C""","""T""",65,4,"""AHCTF1""","""RCC25-1057""","""C25015""","""S_20_18031""","""1""",246902664,246903966,"""N""",20
2162,"""EA700559""","""1""",18875499,18875499,"""Missense_Mutation""","""SNP""","""c.1343T>C""","""c.(1342-1344)aTc>aCc""","""p.I448T""","""A""","""A""","""G""",28,21,"""ALDH4A1""","""RCC25-3""","""C2941""","""S_20_2188""","""1""",18875414,18876356,"""N""",20


In [18]:
braunmut_scaffold10.join(braunpephla_scaffold10, on="Id")

mutid,Sample_ID,Chromosome,Start_position,End_position,Variant_Classification,Variant_Type,cDNA_Change,Codon_Change,Protein_Change,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,Tumor_ref_count,Tumor_alt_count,gene_name,SUBJID,contig,Id,Chromosome_1,Gene_start,Gene_end,Multi_contig,gap_width,Haplotype,pep_list,unique_peptides,n_unique_HLA
u32,str,str,i64,i64,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64,str,i64,list[str],list[str],u32,u32
55176,"""EA700375""","""9""",75133306,75133306,"""Missense_Mutation""","""SNP""","""c.263G>T""","""c.(262-264)tGg>tTg""","""p.W88L""","""G""","""G""","""T""",60,15,"""OSTF1""","""RCC25-1071""","""C114883""","""S_10_95711""","""9""",75131772,75133307,"""N""",10,"[""A*02:01"", ""B*40:02"", … ""A*02:20""]","[""ESIDNPLHEAAKR"", ""AESIDNPLHEA"", … ""ESIDNPLHEAAKR""]",60,60
45160,"""EA699811""","""20""",4699615,4699615,"""Missense_Mutation""","""SNP""","""c.395G>A""","""c.(394-396)aGt>aAt""","""p.S132N""","""G""","""G""","""A""",220,32,"""PRNP""","""RCC25-1055""","""C225931""","""S_10_188318""","""20""",4699578,4699802,"""N""",10,"[""A*23:01"", ""A*24:02"", … ""B*13:02""]","[""HRYPNQVYY"", ""AVVGGLGGY"", … ""HRYPNQVYY""]",213,213
1129,"""RP-1458_RCCBMS-00119-T_v2_Exom…","""16""",67882496,67882496,"""Missense_Mutation""","""SNP""","""c.3344G>A""","""c.(3343-3345)cGg>cAg""","""p.R1115Q""","""G""","""G""","""A""",36,14,"""EDC4""","""RCC10-119""","""C187845""","""S_10_156470""","""16""",67882468,67882536,"""N""",10,"[""A*32:01"", ""B*07:02"", … ""C*03:04""]","[""REAFQSVVL"", ""LQGPMQAAY"", … ""REAFQSVVL""]",74,74
16279,"""EA699745""","""7""",985018,985018,"""Missense_Mutation""","""SNP""","""c.406C>A""","""c.(406-408)Ctg>Atg""","""p.L136M""","""C""","""C""","""A""",167,19,"""CYP2W1""","""RCC25-451""","""C91752""","""S_10_76471""","""7""",984994,985053,"""Y""",10,"[""B*40:01"", ""A*29:02"", … ""A*32:01""]","[""FTVRALHSL"", ""FTVRALHSL"", … ""FTVRALHSL""]",14,14
54440,"""EA700159""","""16""",84659928,84659928,"""Intron""","""SNP""",null,null,null,"""G""","""G""","""A""",118,5,"""KLHL36""","""RCC25-846""","""C189744""","""S_10_158021""","""16""",84657888,84661587,"""N""",10,"[""B*27:05"", ""A*32:01"", … ""B*07:02""]","[""SYVAGLPRF"", ""YVAGLPRFTY"", … ""YVAGLPRFTY""]",108,108
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
58102,"""RP-1458_RCCBMS-00136-T_v1_Exom…","""16""",70571730,70571730,"""Missense_Mutation""","""SNP""","""c.3571A>G""","""c.(3571-3573)Aag>Gag""","""p.K1191E""","""A""","""A""","""G""",36,10,"""SF3B3""","""RCC10-136""","""C188622""","""S_10_157079""","""16""",70571724,70571810,"""N""",10,"[""B*15:01"", ""A*02:06"", … ""A*01""]","[""KLEDIRTRY"", ""DIRTRYAF"", … ""DIRTRYAF""]",171,171
57063,"""EA700520""","""15""",28213809,28213809,"""Missense_Mutation""","""SNP""","""c.6719G>T""","""c.(6718-6720)gGc>gTc""","""p.G2240V""","""C""","""C""","""A""",199,42,"""HERC2""","""RCC25-252""","""C171701""","""S_10_143043""","""15""",28213796,28213837,"""N""",10,"[""A*03:01"", ""B*08:01"", … null]","[""TPKGKITV"", ""TPKGKITV"", … ""GTVTRITPK""]",8,8
40721,"""EA699821""","""2""",219418719,219418719,"""Missense_Mutation""","""SNP""","""c.257G>T""","""c.(256-258)gGc>gTc""","""p.G86V""","""G""","""G""","""T""",214,6,"""DES""","""RCC25-5""","""C41239""","""S_10_34165""","""2""",219418466,219421510,"""Y""",10,"[""A*30:01"", ""C*02:02"", … ""B*49:01""]","[""SLADAVNQEF"", ""ALAAEVNRL"", … ""ADAVNQEF""]",2682,2682


In [21]:
scaf10prom

Id,popcov_but_sqrt,popcov_but_sqrt2,popcov_but_sqrt3,popcov_but_sqrt4
str,f64,f64,f64,f64
"""S_10_100""",0.0,0.0,0.0,0.0
"""S_10_1000""",0.376947,0.0,0.0,0.0
"""S_10_100000""",0.57704,0.0,0.0,0.0
"""S_10_100002""",61.040115,0.729108,0.0,0.0
"""S_10_100003""",1.08112,0.0,0.0,0.0
…,…,…,…,…
"""S_10_99987""",17.539472,0.0,0.0,0.0
"""S_10_99988""",0.373833,0.0,0.0,0.0
"""S_10_99989""",0.999627,0.0,0.0,0.0


In [31]:
braunmut.join(braunmut_scaffold10.join(braunpephla_scaffold10, on="Id").drop(["Haplotype", "pep_list"]).join(scaf10prom, on="Id"), on="mutid", how="left").write_csv("Braun_hg38_epscaff10_w_score_2025.tsv", separator="\t")

/tmp/ipykernel_520103/2034034041.py:1: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  braunmut.join(braunmut_scaffold10.join(braunpephla_scaffold10, on="Id").drop(["Haplotype", "pep_list"]).join(scaf10prom, on="Id"), on="mutid", how="left").write_csv("Braun_hg38_epscaff10_w_score_2025.tsv", separator="\t")


In [36]:
braunmut.join(braunmut_scaffold20.join(braunpephla_scaffold20, on="Id").drop(["Haplotype", "pep_list"]).join(scaf20prom, on="Id"), on="mutid", how="left").write_csv("Braun_hg38_epscaff20_w_score_2025.tsv", separator="\t")

/tmp/ipykernel_520103/4088974471.py:1: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  braunmut.join(braunmut_scaffold20.join(braunpephla_scaffold20, on="Id").drop(["Haplotype", "pep_list"]).join(scaf20prom, on="Id"), on="mutid", how="left").write_csv("Braun_hg38_epscaff20_w_score_2025.tsv", separator="\t")


# TODO i don't think the rest is necessary anymore

In [39]:
braunmut_scaffold10.join(braunpephla_scaffold10, on="Id").drop(["Haplotype", "pep_list"]).join(scaf10prom, on="Id").write_csv("Braun_hg38_epscaff10_w_score.tsv", separator="\t")

In [40]:
braunmut_scaffold20.join(braunpephla_scaffold20, on="Id").drop(["Haplotype", "pep_list"]).join(scaf20prom, on="Id").write_csv("Braun_hg38_epscaff20_w_score.tsv", separator="\t")

In [6]:
scaff10 = contig_scaffold_list.filter(pl.col("gap_width")==10).drop("Contigs").unique().with_columns(pl.col("Chromosome").str.strip_chars_start("chr"))

In [7]:
scaff20 = contig_scaffold_list.filter(pl.col("gap_width")==20).drop("Contigs").unique().with_columns(pl.col("Chromosome").str.strip_chars_start("chr"))

In [12]:
braunmut_scaffold20 = duckdb.sql('''
                                 select * 
                                 from braunmut b, scaff20 c
                                 where c.Chromosome = b.Chromosome
                                 and(
                                     (b.Start_position between c.Gene_start and c.Gene_end)
                                   or(b.End_position between c.Gene_start and c.Gene_end)
                                 )
                                 ''').pl()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
braunmut_scaffold20.write_csv("Braun_hg38_mutations_in_scaffolds_20gw.csv")

In [29]:
braunmut_scaffold20

Sample_ID,Chromosome,Start_position,End_position,Variant_Classification,Variant_Type,cDNA_Change,Codon_Change,Protein_Change,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,Tumor_ref_count,Tumor_alt_count,gene_name,SUBJID,contig,Id,Chromosome_1,Gene_start,Gene_end,Multi_contig,gap_width
str,str,i64,i64,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64,str,i64
"""EA700354""","""X""",49251615,49251615,"""Intron""","""SNP""",null,null,null,"""C""","""C""","""T""",294,5,"""FOXP3""","""RCC25-298""","""C241863""","""S_20_174474""","""X""",49251436,49251675,"""Y""",20
"""EA699702""","""12""",6937305,6937305,"""Missense_Mutation""","""SNP""","""c.2038G>C""","""c.(2038-2040)Gca>Cca""","""p.A680P""","""G""","""G""","""C""",213,6,"""ATN1""","""RCC25-783""","""C146000""","""S_20_105346""","""12""",6937290,6937325,"""N""",20
"""EA698476""","""1""",22908007,22908007,"""Missense_Mutation""","""SNP""","""c.2194G>A""","""c.(2194-2196)Gca>Aca""","""p.A732T""","""G""","""G""","""A""",109,9,"""EPHB2""","""RCC25-223""","""C3766""","""S_20_2746""","""1""",22907986,22908012,"""N""",20
"""EA700455""","""2""",151642690,151642690,"""Intron""","""SNP""",null,null,null,"""G""","""G""","""C""",25,13,"""NEB""","""RCC25-139""","""C35873""","""S_20_25666""","""2""",151642667,151642776,"""N""",20
"""EA698900""","""14""",77026709,77026709,"""Missense_Mutation""","""SNP""","""c.1084C>T""","""c.(1084-1086)Cgc>Tgc""","""p.R362C""","""G""","""G""","""A""",222,5,"""IRF2BPL""","""RCC25-408""","""C168452""","""S_20_121542""","""14""",77026647,77026769,"""Y""",20
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""EA698477""","""19""",44781224,44781224,"""Missense_Mutation""","""SNP""","""c.518C>T""","""c.(517-519)gCt>gTt""","""p.A173V""","""C""","""C""","""T""",89,4,"""CBLC""","""RCC25-360""","""C220229""","""S_20_158741""","""19""",44781214,44781240,"""N""",20
"""EA699646""","""19""",5667839,5667839,"""Silent""","""SNP""","""c.2577C>G""","""c.(2575-2577)tcC>tcG""","""p.S859S""","""C""","""C""","""G""",128,38,"""SAFB""","""RCC25-582""",null,"""S_20_151987""","""19""",5667378,5667866,"""Y""",20
"""EA699646""","""1""",204443421,204443421,"""Missense_Mutation""","""SNP""","""c.3044G>A""","""c.(3043-3045)aGg>aAg""","""p.R1015K""","""C""","""C""","""T""",41,9,"""PIK3C2B""","""RCC25-582""","""C21102""","""S_20_15245""","""1""",204442586,204443470,"""N""",20
